# B2 — frozen foundation embeddings (SwinUNETR SSL)

Upload `trackc.py` **and** `b2_foundation.py` next to `data/`, then Run All.
This tests whether the foundation embedding beats the gradient baseline on the held-out 50 (Level-1) and its deformed copy (Level-2 proxy). If yes, we wire it into RRF and extract all 6 pools.

Paste back: the **SSL load** line, and the **L1 / L2 MRR** numbers for baseline vs B2.

In [ ]:
import importlib, trackc, b2_foundation as b2
importlib.reload(trackc); importlib.reload(b2)
from trackc import *
import numpy as np, pandas as pd

DATA_ROOT = 'data'
trackc.DATA_ROOT = DATA_ROOT
manifest = load_manifest(DATA_ROOT)
train_df, hold_df, gt = make_local_split(manifest['train_pairs'], n_holdout=50, seed=0)
print('held-out pairs:', len(hold_df))

## 1. Download weights + build the frozen encoder (watch the 'SSL load: matched X/Y' line)

In [ ]:
b2.download_weights()
model, device = b2.build_encoder()
print('device:', device)

## 2. Embed the held-out 50 queries + targets, measure Level-1 MRR

In [ ]:
qE = b2.embed_dataframe(model, device, hold_df, 'query_id', 'query_image')
gE = b2.embed_dataframe(model, device, hold_df, 'target_id', 'target_image')
rank_b2_l1 = rank_by_embeddings(qE, gE)
print('B2  Level-1 local MRR =', round(mrr(rank_b2_l1, gt), 4))

## 3. Level-2 proxy: deform each held-out target, re-embed, re-measure
The key test — does a deep encoder survive deformation better than raw gradients?

In [ ]:
import torch
rng = np.random.default_rng(0)
gE_l2 = {}
for i, (_, r) in enumerate(hold_df.iterrows()):
    vol = load_volume(r['target_image'], resample_1mm=True, zscore=True, data_root=DATA_ROOT)
    vol = resize_to(vol, b2.INPUT_SIZE)
    vol = deform_volume(vol, rng=rng)
    x = torch.from_numpy(vol)[None, None].float().to(device)
    with torch.no_grad():
        out = model.swinViT(x)
        feat = out[-1] if isinstance(out, (list, tuple)) else out
        vec = torch.nn.functional.adaptive_avg_pool3d(feat, 1).flatten().cpu().numpy()
    gE_l2[r['target_id']] = (vec / (np.linalg.norm(vec) + 1e-8)).astype(np.float32)
    if (i + 1) % 25 == 0: print('  ', i + 1, '/', len(hold_df))
rank_b2_l2 = rank_by_embeddings(qE, gE_l2)
print('B2  Level-2 proxy local MRR =', round(mrr(rank_b2_l2, gt), 4))

## 4. Verdict vs baseline (from run_pipeline.ipynb: L1=0.7626, L2=0.0981)

In [ ]:
print('             L1      L2(deform)')
print('baseline   0.7626   0.0981')
print(f'B2 (Swin)  {mrr(rank_b2_l1, gt):.4f}   {mrr(rank_b2_l2, gt):.4f}')
print('\n-> if B2 adds signal (esp. on L2), we extract all 6 pools and fuse via RRF.')